In [ ]:
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!pip install -r requirements.txt
!pip install kornia==0.7.1 onnxruntime-gpu insightface
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git custom_nodes/ComfyUI-Manager
!git clone https://github.com/Lightricks/ComfyUI-LTXVideo.git custom_nodes/ComfyUI-LTXVideo
!pip install -r custom_nodes/ComfyUI-LTXVideo/requirements.txt
!git clone https://github.com/Gourieff/ComfyUI-ReActor.git custom_nodes/ComfyUI-ReActor
!pip install -r custom_nodes/ComfyUI-ReActor/requirements.txt
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
import os
print("[1/5] Downloading DreamShaper 8 (Text-to-Image)...")
!mkdir -p models/checkpoints
!wget -q -O models/checkpoints/dreamshaper_8.safetensors https://huggingface.co/Lykon/dreamshaper-8/resolve/main/dreamshaper_8.safetensors
print("[2/5] Downloading LTX-Video 0.9.1 (Image-to-Video)...")
!mkdir -p models/checkpoints/LTX-Video
!wget -q -O models/checkpoints/LTX-Video/ltx-video-2b-v0.9.1.safetensors https://huggingface.co/Lightricks/LTX-Video/resolve/main/ltx-video-2b-v0.9.1.safetensors
print("[3/5] Downloading T5 FP8 Text Encoder...")
!mkdir -p models/clip
!wget -q -O models/clip/t5xxl_fp8_e4m3fn.safetensors https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors
print("[4/5] Downloading CodeFormer (Face Restoration)...")
!mkdir -p models/facerestore_models
!wget -q -O models/facerestore_models/codeformer-v0.1.0.pth https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/codeformer.pth
print("[5/5] Copying InsightFace models from Kaggle Dataset...")
!mkdir -p models/insightface
!find /kaggle/input -name "inswapper_128.onnx" -exec cp {} models/insightface/inswapper_128.onnx \;
print("All models downloaded!")

In [ ]:
import subprocess
import threading
import time
import socket

def tunnel_thread():
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', 8188))
        if result == 0:
            break
        sock.close()
    print("\n\ud83d\ude80 Launching Cloudflare Tunnel...\n")
    p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com " in l:
            print("\n\n\u2728 ComfyUI is LIVE! Click this link to open: ", l[l.find("http"):], "\n\n")

threading.Thread(target=tunnel_thread, daemon=True).start()

# Start ComfyUI on 0.0.0.0 and allow all CORS to prevent localhost proxy blocks
!python main.py --fp8_e4m3fn-unet --fp8_e4m3fn-text-enc --listen 0.0.0.0 --enable-cors-header "*"